# Phase 2 特徵工程測試與驗證（Multi-timeframe）\n\n本 Notebook 用於：\n- 逐一讀取每個指標的 checkpoint 檔案（roll / roll_impact / amihud / kyle_lambda / vpin）\n- 檢查欄位命名是否符合規範（5 窗口：50 / 100 / 240 / 480 / 720）\n- 驗證各窗口前 W-1 筆為 NaN\n- 計算 NaN 比率、基本統計摘要（mean/std/min/max）\n- 繪製多時間框架對比圖（例如 roll_50 vs roll_720）\n\n資料來源（由 `FeaturePaths.default()` / `DATA_LAKE_ROOT` 解析，預設 `C:\\Data_Lake`）：\n- 輸入 raw：`{DATA_LAKE}/raw/crypto/binance/{symbol}_1m_2021-2026.parquet`\n- 輸出 features：`{DATA_LAKE}/processed/crypto_microstructure/features/{symbol}/...`\n

In [ ]:
import os\n
import sys\n
from pathlib import Path\n
\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
\n
# 若你 Notebook 啟動目錄不是專案根目錄，請調整這段\n
ROOT = Path(os.getcwd()).resolve().parent\n
if str(ROOT) not in sys.path:\n
    sys.path.insert(0, str(ROOT))\n
\n
print('Project root =', ROOT)\n
print('DATA_LAKE_ROOT =', os.environ.get('DATA_LAKE_ROOT', '(未設定，將使用 utils/config.py 預設)'))

## 讀取工具與路徑

In [ ]:
from features.feature_utils import FeaturePaths, load_feature\n
from utils.config import SYMBOLS\n
\n
paths = FeaturePaths.default()\n
print('Features root =', paths.root)\n
print('Symbols =', SYMBOLS)

## 逐一指標讀取與統計摘要

In [ ]:
INDICATORS = [\n
    ('roll', ['roll_50', 'roll_100']),\n
    ('roll_impact', ['roll_impact_50', 'roll_impact_100']),\n
    ('amihud', ['amihud_50', 'amihud_100']),\n
    ('kyle_lambda', ['kyle_lambda_50', 'kyle_lambda_100']),\n
    ('vpin', ['vpin_50', 'vpin_100']),\n
]\n
\n
def summarize(df: pd.DataFrame) -> pd.DataFrame:\n
    out = []\n
    for c in df.columns:\n
        s = df[c]\n
        out.append({\n
            'col': c,\n
            'nan_ratio': float(s.isna().mean()),\n
            'mean': float(s.mean(skipna=True)),\n
            'std': float(s.std(skipna=True)),\n
            'min': float(s.min(skipna=True)),\n
            'max': float(s.max(skipna=True)),\n
        })\n
    return pd.DataFrame(out)\n
\n
for sym in SYMBOLS:\n
    print('\n====', sym, '====')\n
    for ind, expected_cols in INDICATORS:\n
        df = load_feature(sym, ind)\n
        missing = [c for c in expected_cols if c not in df.columns]\n
        if missing:\n
            print(f'❌ {ind} 缺少欄位：{missing}')\n
        else:\n
            print(f'✅ {ind} 欄位 OK：{expected_cols}')\n
        display(df.head())\n
        display(summarize(df))\n

## 多時間框架對比圖（roll_50 vs roll_720）

In [ ]:
sym = SYMBOLS[0]\n
ind = 'vpin'\n
df = load_feature(sym, ind)\n
\n
# 取最後 7 天（約 10080 分鐘）避免畫太慢\n
tail = df.tail(60 * 24 * 7)\n
\n
ax = tail.plot(figsize=(12, 4), title=f'{sym} - {ind} (last 7 days)')\n
ax.set_xlabel('open_time')\n
ax.set_ylabel(ind)\n
plt.show()

## combined_features 檢查（若你已執行 feature_pipeline.py）

In [ ]:
from features.feature_utils import FeaturePaths\n
\n
paths = FeaturePaths.default()\n
for sym in SYMBOLS:\n
    p = paths.combined_path(sym)\n
    if not p.is_file():\n
        print(f'⏭️ {sym} 尚未產生 combined_features.parquet：{p}')\n
        continue\n
    combined = pd.read_parquet(p)\n
    print(f'✅ {sym} combined_features 讀取成功：shape={combined.shape}')\n
    display(combined.head())\n
    display(combined.isna().mean().to_frame('nan_ratio').T)